# lean_hedged: end-to-end submission

Builds the `lean_hedged` submission (Kaggle public **60.66**) stage by stage:

1. **Load and validate** the competition files
2. **Features**: news-story parser, supplier state, market z-scores
3. **Disruption model**: GBM onset classifier + story-script model + duration model
4. **Price model**: shock-free level, mean reversion, shock uplift
5. **Ordering**: stochastic-programming LP (30 scenarios, exact simulator rules)
6. **Hedge**: calm-bet forecast columns (this file is the calm half of the final pair)
7. **Validate and write** the CSV

Paths come from `DSDG_DATA_DIR` (default `data/comp`), so this also runs on Databricks with the data in a Volume.
Every choice below is logged in `DECISIONS.md`.

In [1]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))           # use the repo's package even without `uv sync`
os.environ.setdefault("DSDG_DATA_DIR", str(ROOT / "data" / "comp"))

import numpy as np
import pandas as pd
pd.set_option("display.width", 200, "display.max_columns", 20)

from dsdg_supply_chain.data import load_data, SHARE_COLS
from dsdg_supply_chain.submission import FIRST_TEST_WEEK, LAST_TEST_WEEK, validate, write_submission, read_submission
print("data dir:", os.environ["DSDG_DATA_DIR"])

data dir: /Users/shubeekshk/Documents/Projects/DSDG-Supply-Chain-Hackathon-2026/data/comp


## 1. Load and validate the data

In [2]:
data = load_data()
WEEKS = range(FIRST_TEST_WEEK, LAST_TEST_WEEK + 1)      # 209..234, always the submission window
ORIGIN = data.last_week                                 # information week: 208 in Phase 1, 221 in Phase 2
for name in ["demand", "news", "outcomes", "prices", "supply_history", "tariffs", "suppliers"]:
    df = getattr(data, name)
    print(f"{name:15s} rows {len(df):6d}  nulls {int(df.isna().sum().sum()):5d}  dup rows {int(df.duplicated().sum())}")
print("information week:", ORIGIN)

demand          rows   2340  nulls     0  dup rows 0
news            rows   4317  nulls     0  dup rows 0
outcomes        rows   2080  nulls     0  dup rows 0
prices          rows   6240  nulls     0  dup rows 0
supply_history  rows   6240  nulls     0  dup rows 0
tariffs         rows   6240  nulls 12420  dup rows 0
suppliers       rows     30  nulls     0  dup rows 0
information week: 208


## 2. Features
One row per (week, material, supplier): news-story stage per shock type, current hit state, contagion
(share of the lane / country already hit) and market z-scores.

In [3]:
from dsdg_supply_chain.features.build import build_features
feats = build_features(data)
print(feats.shape)
feats[feats.week == ORIGIN].query("news_stage > 0 or hit == 1")[["material", "country", "hit", "news_type", "news_stage", "story_age"]]

(6240, 45)


,material,country,hit,news_type,news_stage,story_age
6214,cobalt,ORI,0,strike,2,1.0
6217,copper,ORI,0,strike,2,1.0
6219,lithium,KES,0,shortage,1,1.0
6220,lithium,ORI,0,strike,2,1.0
6223,natural_gas,ORI,0,strike,2,1.0


## 3. Disruption model
* **Onset**: GBM classifier for suppliers not hit now (news + contagion features).
* **Story scripts** (`models/scripts.py`): for every live news story,
  P(hit) = conversion(type, stage) × lead distribution × duration, learned only from stories resolved before the origin.
  Combined with the GBM as max(GBM, script).
* **Duration**: censoring-aware log-normal per shock type for suppliers hit now.
* Material probability = 1 − Π(1 − p_supplier), blended toward the historical rate.

In [4]:
from dsdg_supply_chain.models.scripts import ScriptModel
sm = ScriptModel.fit(data)
print("conversion rates (type, stage):", {k: round(v, 2) for k, v in sm.conv.items() if k[1] == 1})
sm.open_stories[["type", "entity", "start", "last", "stage"]]

conversion rates (type, stage): {('ban', 1): 0.72, ('closure', 1): 0.38, ('reversal', 1): 0.65, ('sanctions', 1): 0.83, ('shortage', 1): 0.86, ('strike', 1): 0.47, ('tariff', 1): 0.86}


,type,entity,start,last,stage
16,shortage,KES/lithium,207,208,1
24,strike,SOUTH,207,208,2


In [5]:
from dsdg_supply_chain.models.disruption import supplier_forecast, predict_disruption
sup = supplier_forecast(data, WEEKS, feats=feats)
mat_p = predict_disruption(data, WEEKS, sup=sup)
mat_p[mat_p.week > ORIGIN].pivot(index="material", columns="week", values="p").round(2).iloc[:, :13]

week,209,210,211,212,213,214,215,216,217,218,219,220,221
material,,,,,,,,,,,,,
aluminum,0.02,0.03,0.04,0.05,0.07,0.08,0.09,0.10,0.12,0.13,0.14,0.14,0.16
cobalt,0.35,0.43,0.38,0.30,0.23,0.18,0.15,0.15,0.13,0.11,0.11,0.11,0.11
copper,0.32,0.41,0.35,0.27,0.20,0.15,0.11,0.11,0.09,0.07,0.07,0.07,0.06
lithium,0.41,0.53,0.71,0.81,0.78,0.73,0.67,0.61,0.53,0.46,0.40,0.35,0.30
natural_gas,0.32,0.41,0.35,0.27,0.20,0.15,0.11,0.11,0.09,0.07,0.07,0.07,0.07
polymer_resin,0.02,0.03,0.04,0.05,0.07,0.08,0.09,0.10,0.11,0.13,0.14,0.14,0.15
rare_earths,0.04,0.04,0.05,0.06,0.07,0.09,0.09,0.10,0.11,0.12,0.13,0.14,0.15
semiconductors,0.02,0.03,0.04,0.05,0.07,0.08,0.09,0.10,0.12,0.13,0.14,0.14,0.15
specialty_glass,0.01,0.02,0.03,0.03,0.05,0.06,0.07,0.08,0.09,0.10,0.11,0.12,0.12


## 4. Price model

In [6]:
from dsdg_supply_chain.models.price import predict_price
px = predict_price(data, WEEKS, sup, feats=feats)
px[px.week > ORIGIN].pivot(index="material", columns="week", values="price").round(2).iloc[:, [0, 4, 8, 12]]

week,209,213,217,221
material,,,,
aluminum,3.07,3.05,3.03,3.01
cobalt,63.47,62.77,62.13,61.49
copper,11.12,11.15,11.18,11.21
lithium,47.33,52.65,52.04,45.69
natural_gas,4.34,4.36,4.38,4.39
polymer_resin,1.75,1.76,1.76,1.77
rare_earths,108.86,108.33,107.83,107.34
semiconductors,590.42,587.95,585.63,583.34
specialty_glass,7.47,7.47,7.48,7.48


## 5. Ordering: stochastic-programming LP
Per material and 13-week window: 30 scenarios drawn from the supplier forecasts, one LP that minimises
the average simulated cost under the competition's rules (fill up to capacity, lead times, 5-week starting
stock, 2% holding, 5× unmet penalty, 70% salvage). Lean settings: no minimum share, no background shocks,
plan at 96% of normal capacity.

In [7]:
from dsdg_supply_chain.strategies import naive, _apply_forecasts
from dsdg_supply_chain.policy_lp import LPParams, plan_orders_lp

sub = _apply_forecasts(naive(data, WEEKS), mat_p, px)            # skeleton + forecast columns
starts = [s for s in range(WEEKS[0], WEEKS[-1] + 1, 13) if s > ORIGIN]
plan_weeks = range(starts[0], WEEKS[-1] + 1)                     # only windows fully in the future
LEAN = LPParams(n_scenarios=30, min_share=0.0, hazard_mult=0.0, cap_derate=0.96)
plan = plan_orders_lp(data, plan_weeks, feats, sup[sup.week > ORIGIN], LEAN).set_index(["week", "material"])

cols = ["order_units", *SHARE_COLS]
sub = sub.set_index(["week", "material"])
sub.loc[plan.index, cols] = plan[cols]
sub = sub.reset_index()[["id", "week", "material", *[c for c in sub.columns if c not in ("id",)]]]
print("plan covers weeks", plan_weeks[0], "-", plan_weeks[-1])
sub[sub.material == "lithium"].set_index("week")[["order_units", "share_KES", "share_VAL", "share_ORI"]].round(2).T

plan covers weeks 209 - 234


week,209,210,211,212,213,214,215,216,217,218,...,225,226,227,228,229,230,231,232,233,234
order_units,806.4,806.4,806.4,1010.2,1284.00,1238.00,1277.00,1261.00,1346.0,441.6,...,1188.00,1205.00,1175.00,1098.00,1091.00,1136.00,288.6,0.00,0.00,0.00
share_KES,1.0,1.0,1.0,0.8,0.63,0.65,0.63,0.64,0.6,0.0,...,0.68,0.67,0.69,0.73,0.74,0.71,0.0,0.33,0.33,0.33
share_VAL,0.0,0.0,0.0,0.2,0.37,0.35,0.37,0.36,0.4,1.0,...,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.33,0.33,0.33
share_ORI,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,0.0,0.0,...,0.32,0.33,0.31,0.27,0.26,0.29,1.0,0.33,0.33,0.33


## 6. Hedge (calm bet)
Kaggle keeps the better of the two final files. This is the calm half: background and ongoing-shock
probabilities are scaled by 0.5 beyond week 2. Live warned stories are left alone.

In [8]:
from dsdg_supply_chain.strategies import hedge_forecasts, story_materials, CALM_A, CALM_K
print("live-story materials (not hedged):", sorted(story_materials(data)), "| CALM_A", CALM_A, "CALM_K", CALM_K)
sub = hedge_forecasts(sub, data, "calm")

live-story materials (not hedged): ['cobalt', 'copper', 'lithium', 'natural_gas'] | CALM_A 0.5 CALM_K 2


## 7. Validate and write

In [9]:
errs = validate(sub, data, WEEKS)
assert not errs, errs
out = ROOT / "submissions" / f"lean_hedged_w{ORIGIN}_nb.csv"
write_submission(sub, out)
print("wrote", out)

ref = ROOT / "submissions" / f"lean_hedged_w{ORIGIN}.csv"   # file built by `dsdg submit lean_hedged`
if ref.exists():
    a, b = pd.read_csv(out), pd.read_csv(ref)
    num = [c for c in a if c != "id"]
    print("identical to", ref.name, ":", a.id.equals(b.id) and np.allclose(a[num], b[num]))

wrote /Users/shubeekshk/Documents/Projects/DSDG-Supply-Chain-Hackathon-2026/submissions/lean_hedged_w208_nb.csv
identical to lean_hedged_w208.csv : True
